In [11]:
import numpy as np
from scipy.optimize import linprog

# 各仓库的化肥实际存量（万吨）
warehouse_stocks = np.array([85, 120, 105])
# 京津冀地区的化肥需求量（万吨）
city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])
# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])
# 车辆信息
vehicle_info = {
    '小型车': {'最大运量': 10, '单位交通成本': 0.145, '单位油耗': 1.4},
    '中型车': {'最大运量': 20, '单位交通成本': 0.0975, '单位油耗': 3.6},
    '重型车': {'最大运量': 40, '单位交通成本': 0.045, '单位油耗': 6.8}
}

# 变量数量：每个仓库-地区组合对应三种车型的运输量，共3*仓库数*地区数
num_vars = 3 * len(warehouse_stocks) * len(city_demands)
# 构建目标函数系数：总运输成本最小化，权重1；总运输车次最小化，权重0.5；平均供给率最大化，权重 -1（转化为最小化负数）
c_transport = []
c_trips = []
c_supply_rate = []
for i in range(len(warehouse_stocks)):
    for j in range(len(city_demands)):
        for vehicle in ['小型车', '中型车', '重型车']:
            dist = distances[i][j]
            # 使用新的成本计算公式：单位运输成本*距离*运量 + 单位油耗*距离
            cost = vehicle_info[vehicle]['单位交通成本'] * dist + vehicle_info[vehicle]['单位油耗'] * dist
            c_transport.append(cost)
            c_trips.append(1 / vehicle_info[vehicle]['最大运量'])
            c_supply_rate.append(-1 / city_demands[j])
c = np.array(c_transport) + 0.5 * np.array(c_trips) - np.array(c_supply_rate)

# 构建不等式约束矩阵A_ub和向量b_ub
A_ub = []
b_ub = []
# 每个仓库发货量不超过库存
for i in range(len(warehouse_stocks)):
    row = [0] * num_vars
    for j in range(len(city_demands)):
        for k in range(3):
            row[i * 3 * len(city_demands) + j * 3 + k] = 1
    A_ub.append(row)
    b_ub.append(warehouse_stocks[i])
# 每个地区分配量不超过需求量
for j in range(len(city_demands)):
    row = [0] * num_vars
    for i in range(len(warehouse_stocks)):
        for k in range(3):
            row[i * 3 * len(city_demands) + j * 3 + k] = 1
    A_ub.append(row)
    b_ub.append(city_demands[j])

# 构建等式约束矩阵A_eq和向量b_eq（所有库存都要分配出去）
A_eq = []
b_eq = []
for i in range(len(warehouse_stocks)):
    row = [0] * num_vars
    for j in range(len(city_demands)):
        for k in range(3):
            row[i * 3 * len(city_demands) + j * 3 + k] = 1
    A_eq.append(row)
b_eq = warehouse_stocks

# 求解线性规划问题
bounds = [(0, None)] * num_vars
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

if res.status == 0:
    optimal_solution = res.x.reshape(len(warehouse_stocks), len(city_demands), 3)
    print('最优配送方案（万吨）：')
    for i, warehouse in enumerate(['仓库A(玉田)', '仓库B(辛集)', '仓库C(南宫)']):
        print(f'{warehouse}:')
        for j, city in enumerate(['石家庄', '唐山', '秦皇岛', '邯郸', '邢台', '保定', '张家口', '承德', '沧州', '廊坊', '衡水', '北京', '天津']):
            print(f'\t{city}: 小型车运量 {optimal_solution[i][j][0]:.6f} 万吨, 中型车运量 {optimal_solution[i][j][1]:.6f} 万吨, 重型车运量 {optimal_solution[i][j][2]:.6f} 万吨')
    
    # 计算总运输成本
    cost_matrix = np.array([[[vehicle_info[vehicle]['单位交通成本'] * distances[i][j] + vehicle_info[vehicle]['单位油耗'] * distances[i][j] 
                              for vehicle in ['小型车', '中型车', '重型车']] 
                             for j in range(len(city_demands))] 
                            for i in range(len(warehouse_stocks))])
    total_transport_cost = np.sum(optimal_solution * cost_matrix)
    
    print(f'最小总运输成本（元）：{total_transport_cost:.2f}')
    
    # 计算总运输车次
    total_trips = np.sum(np.ceil(optimal_solution / np.array([[[vehicle_info[vehicle]['最大运量'] for vehicle in ['小型车', '中型车', '重型车']] 
                                                                 for _ in range(len(city_demands))] 
                                                                for _ in range(len(warehouse_stocks))])))
    print(f'总运输车次（取整）：{int(total_trips)}')
    
    # 计算各目标地化肥市场平均供给率
    total_supply = np.sum(optimal_solution, axis=(0, 2))
    average_supply_rate = np.mean(total_supply / city_demands)
    print(f'各目标地化肥市场平均供给率：{average_supply_rate:.4f}')
    
    # 输出各地区的具体供给率
    print('\n各地区供给率：')
    for j, city in enumerate(['石家庄', '唐山', '秦皇岛', '邯郸', '邢台', '保定', '张家口', '承德', '沧州', '廊坊', '衡水', '北京', '天津']):
        supply_rate = total_supply[j] / city_demands[j]
        print(f'{city}: {supply_rate:.4f}')
else:
    print(f'线性规划求解失败，状态码: {res.status}')
    if res.status == 1:
        print('超过了迭代次数限制')
    elif res.status == 2:
        print('问题不可行')
    elif res.status == 3:
        print('目标函数无界')
    elif res.status == 4:
        print('算法特定的错误')


最优配送方案（万吨）：
仓库A(玉田):
	石家庄: 小型车运量 0.000000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	唐山: 小型车运量 30.175434 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	秦皇岛: 小型车运量 8.575503 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	邯郸: 小型车运量 0.000000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	邢台: 小型车运量 0.000000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	保定: 小型车运量 0.000000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	张家口: 小型车运量 0.000000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	承德: 小型车运量 14.959592 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	沧州: 小型车运量 0.000000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	廊坊: 小型车运量 14.419471 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	衡水: 小型车运量 0.000000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	北京: 小型车运量 10.530000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	天津: 小型车运量 6.340000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
仓库B(辛集):
	石家庄: 小型车运量 31.538220 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	唐山: 小型车运量 0.000000 万吨, 中型车运量 0.000000 万吨, 重型车运量 0.000000 万吨
	秦皇岛: 小型车运量 0.000000 万吨, 中型车运量 0.000000 万吨, 重型

In [1]:
import numpy as np
from itertools import product
from math import ceil

# 初始方案的最优配送结果（万吨）
optimal_solution = np.array([
    [ 0.        , 30.175433907508,  9.06583095394121,  0.        ,  0.        ,  0.        ,
      0.        , 14.9595917197154,  0.        , 13.9291434188354,  0.        , 10.53     ,
      6.34      ],
    [31.5382204877364,  0.        ,  0.        ,  0.        ,  0.        , 43.3839807620296,
      0.        ,  0.        , 39.422623689384,  5.65517506085007,  0.        ,  0.        ,
      0.        ],
    [ 0.        ,  0.        ,  0.        , 36.9777190601011, 36.911715151657,  0.        ,
      0.        ,  0.        ,  0.        ,  0.725472406852631, 30.3850933813893,  0.        ,
      0.        ]
])*10000

# 车型信息
vehicle_info = {
    '小型车': {'最大运量': 10, '单位交通成本': 0.145, '单位油耗': 1.4},
    '中型车': {'最大运量': 20, '单位交通成本': 0.0975, '单位油耗': 3.6},
    '重型车': {'最大运量': 40, '单位交通成本': 0.045, '单位油耗': 6.8}
}

# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])

vehicle_types = list(vehicle_info.keys())

print('车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：')
total_trips = 0
total_cost = 0

for i, warehouse in enumerate(['仓库A(玉田)', '仓库B(辛集)', '仓库C(南宫)']):
    print(f'\n{warehouse}:')
    for j, city in enumerate(['石家庄', '唐山', '秦皇岛', '邯郸', '邢台', '保定', '张家口', '承德', '沧州', '廊坊', '衡水', '北京', '天津']):
        if optimal_solution[i, j] > 0:
            print(f'  到{city} ({optimal_solution[i, j]:.2f}万吨):')
            weight = optimal_solution[i, j]
            distance = distances[i][j]
            
            # 计算每种车型最多需要的数量
            max_vehicles = {}
            for vehicle in vehicle_types:
                max_vehicles[vehicle] = ceil(weight / vehicle_info[vehicle]['最大运量'])
            
            # 生成所有可能的车型组合
            possible_combinations = []
            for combo in product(
                range(max_vehicles[vehicle_types[0]] + 1),
                range(max_vehicles[vehicle_types[1]] + 1),
                range(max_vehicles[vehicle_types[2]] + 1)
            ):
                # 计算该组合的总运力
                total_capacity = (
                    combo[0] * vehicle_info[vehicle_types[0]]['最大运量'] + 
                    combo[1] * vehicle_info[vehicle_types[1]]['最大运量'] + 
                    combo[2] * vehicle_info[vehicle_types[2]]['最大运量']
                )
                
                # 如果总运力大于等于需要运输的重量，则该组合有效
                if total_capacity >= weight:
                    possible_combinations.append(combo)
            
            # 计算每种组合的总成本
            min_cost = float('inf')
            best_combo = None
            
            for combo in possible_combinations:
                remaining = weight
                cost = 0
                
                # 按车型容量从大到小分配
                vehicles_used = {
                    vehicle_types[0]: combo[0],
                    vehicle_types[1]: combo[1],
                    vehicle_types[2]: combo[2]
                }
                
                # 先使用重型车
                if vehicles_used['重型车'] > 0:
                    for _ in range(vehicles_used['重型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['重型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['重型车']['单位交通成本'] * distance * load +
                            vehicle_info['重型车']['单位油耗'] * distance
                        )
                        remaining -= load
                
                # 再使用中型车
                if vehicles_used['中型车'] > 0:
                    for _ in range(vehicles_used['中型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['中型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['中型车']['单位交通成本'] * distance * load +
                            vehicle_info['中型车']['单位油耗'] * distance
                        )
                        remaining -= load
                
                # 最后使用小型车
                if vehicles_used['小型车'] > 0:
                    for _ in range(vehicles_used['小型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['小型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['小型车']['单位交通成本'] * distance * load +
                            vehicle_info['小型车']['单位油耗'] * distance
                        )
                        remaining -= load
                
                # 更新最优方案
                if cost < min_cost:
                    min_cost = cost
                    best_combo = combo
            
            # 输出最优方案
            if best_combo:
                total_cost += min_cost
                total_trips += sum(best_combo)
                
                print('    最优方案:')
                for idx, vehicle in enumerate(vehicle_types):
                    if best_combo[idx] > 0:
                        print(f'    {vehicle}: {best_combo[idx]}车次')
                
                print(f'    成本: {min_cost:.2f}万元')

print(f'\n总运输车次: {total_trips}')
print(f'总运输成本: {total_cost:.2f}万元')


车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：

仓库A(玉田):
  到唐山 (301754.34万吨):


: 

In [ ]:
# 微调运输方案1
import numpy as np
from itertools import product
from math import ceil

# 初始方案的最优配送结果（万吨）
optimal_solution = np.array([
    [ 0.        , 30,  9.06583095394121,  0.        ,  0.        ,  0.        ,
      0.        , 14.9595917197154,  0.        , 14.6345773263434,  0.        , 10     ,
      6.34      ],
    [30,  0.        ,  0.        ,  0.        ,  0.        , 43.3839807620296,
      0.        ,  0.        , 39.9129516944393,  5.6752135601947,  0.        ,  0.        ,
      0.        ],
    [ 0.        ,  0.        ,  0.        , 36.9777190601011, 36.911715151657,  0.        ,
      0.        ,  0.        ,  0.        ,  0, 30,  0.        ,
      0.        ]
])

# 车型信息
vehicle_info = {
    '小型车': {'最大运量': 10, '单位交通成本': 0.145, '单位油耗': 1.4},
    '中型车': {'最大运量': 20, '单位交通成本': 0.0975, '单位油耗': 3.6},
    '重型车': {'最大运量': 40, '单位交通成本': 0.045, '单位油耗': 6.8}
}

# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])

vehicle_types = list(vehicle_info.keys())

print('车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：')
total_trips = 0
total_cost = 0

for i, warehouse in enumerate(['仓库A(玉田)', '仓库B(辛集)', '仓库C(南宫)']):
    print(f'\n{warehouse}:')
    for j, city in enumerate(['石家庄', '唐山', '秦皇岛', '邯郸', '邢台', '保定', '张家口', '承德', '沧州', '廊坊', '衡水', '北京', '天津']):
        if optimal_solution[i, j] > 0:
            print(f'  到{city} ({optimal_solution[i, j]:.2f}万吨):')
            weight = optimal_solution[i, j]
            distance = distances[i][j]

            # 计算每种车型最多需要的数量
            max_vehicles = {}
            for vehicle in vehicle_types:
                max_vehicles[vehicle] = ceil(weight / vehicle_info[vehicle]['最大运量'])

            # 生成所有可能的车型组合
            possible_combinations = []
            for combo in product(
                range(max_vehicles[vehicle_types[0]] + 1),
                range(max_vehicles[vehicle_types[1]] + 1),
                range(max_vehicles[vehicle_types[2]] + 1)
            ):
                # 计算该组合的总运力
                total_capacity = (
                    combo[0] * vehicle_info[vehicle_types[0]]['最大运量'] +
                    combo[1] * vehicle_info[vehicle_types[1]]['最大运量'] +
                    combo[2] * vehicle_info[vehicle_types[2]]['最大运量']
                )

                # 如果总运力大于等于需要运输的重量，则该组合有效
                if total_capacity >= weight:
                    possible_combinations.append(combo)

            # 计算每种组合的总成本
            min_cost = float('inf')
            best_combo = None

            for combo in possible_combinations:
                remaining = weight
                cost = 0

                # 按车型容量从大到小分配
                vehicles_used = {
                    vehicle_types[0]: combo[0],
                    vehicle_types[1]: combo[1],
                    vehicle_types[2]: combo[2]
                }

                # 先使用重型车
                if vehicles_used['重型车'] > 0:
                    for _ in range(vehicles_used['重型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['重型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['重型车']['单位交通成本'] * distance * load +
                            vehicle_info['重型车']['单位油耗'] * distance
                        )
                        remaining -= load

                # 再使用中型车
                if vehicles_used['中型车'] > 0:
                    for _ in range(vehicles_used['中型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['中型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['中型车']['单位交通成本'] * distance * load +
                            vehicle_info['中型车']['单位油耗'] * distance
                        )
                        remaining -= load

                # 最后使用小型车
                if vehicles_used['小型车'] > 0:
                    for _ in range(vehicles_used['小型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['小型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['小型车']['单位交通成本'] * distance * load +
                            vehicle_info['小型车']['单位油耗'] * distance
                        )
                        remaining -= load

                # 更新最优方案
                if cost < min_cost:
                    min_cost = cost
                    best_combo = combo

            # 输出最优方案
            if best_combo:
                total_cost += min_cost
                total_trips += sum(best_combo)

                print('    最优方案:')
                for idx, vehicle in enumerate(vehicle_types):
                    if best_combo[idx] > 0:
                        print(f'    {vehicle}: {best_combo[idx]}车次')

                print(f'    成本: {min_cost:.2f}万元')

city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])
demand_satisfaction = np.sum(optimal_solution, axis=0) / city_demands

print(f'\n总运输车次: {total_trips}')
print(f'总运输成本: {total_cost:.2f}万元')
print('各地区需求满足率：', demand_satisfaction)
print('平均需求满足率：', np.mean(demand_satisfaction))

车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：

仓库A(玉田):
  到唐山 (30.00万吨):
    最优方案:
    重型车: 1车次
    成本: 519.15万元
  到秦皇岛 (9.07万吨):
    最优方案:
    小型车: 1车次
    成本: 482.10万元
  到承德 (14.96万吨):
    最优方案:
    小型车: 2车次
    成本: 805.00万元
  到廊坊 (14.63万吨):
    最优方案:
    小型车: 2车次
    成本: 679.24万元
  到北京 (10.00万吨):
    最优方案:
    小型车: 1车次
    成本: 387.31万元
  到天津 (6.34万吨):
    最优方案:
    小型车: 1车次
    成本: 285.74万元

仓库B(辛集):
  到石家庄 (30.00万吨):
    最优方案:
    重型车: 1车次
    成本: 658.52万元
  到保定 (43.38万吨):
    最优方案:
    小型车: 1车次
    重型车: 1车次
    成本: 1569.41万元
  到沧州 (39.91万吨):
    最优方案:
    重型车: 1车次
    成本: 1445.86万元
  到廊坊 (5.68万吨):
    最优方案:
    小型车: 1车次
    成本: 606.63万元

仓库C(南宫):
  到邯郸 (36.98万吨):
    最优方案:
    重型车: 1车次
    成本: 1272.14万元
  到邢台 (36.91万吨):
    最优方案:
    重型车: 1车次
    成本: 883.33万元
  到衡水 (30.00万吨):
    最优方案:
    重型车: 1车次
    成本: 448.25万元

总运输车次: 16
总运输成本: 10042.69万元
各地区需求满足率： [0.95122678 0.9941862  1.         1.         1.         1.
 0.         1.         1.         1.         0.98732624 0.94966762
 1.        ]
平均需求

In [ ]:
# 微调运输方案2
import numpy as np
from itertools import product
from math import ceil

# 初始方案的最优配送结果（万吨）
optimal_solution = np.array([
    [ 0.        , 30,  9.06583095394121,  0.        ,  0.        ,  0.        ,
      0.        , 14.9595917197154,  0.        , 14.6345773263434,  0.        , 10     ,
      6.34      ],
    [30,  0.        ,  0.        ,  0.        ,  0.        , 40,
      0.        ,  0.        , 39.9129516944393,  5.6752135601947,  0.        ,  0.        ,
      0.        ],
    [ 0.        ,  0.        ,  0.        , 36.9777190601011, 36.911715151657,  0.        ,
      0.        ,  0.        ,  0.        ,  0, 30,  0.        ,
      0.        ]
])

# 车型信息
vehicle_info = {
    '小型车': {'最大运量': 10, '单位交通成本': 0.145, '单位油耗': 1.4},
    '中型车': {'最大运量': 20, '单位交通成本': 0.0975, '单位油耗': 3.6},
    '重型车': {'最大运量': 40, '单位交通成本': 0.045, '单位油耗': 6.8}
}

# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])

vehicle_types = list(vehicle_info.keys())

print('车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：')
total_trips = 0
total_cost = 0

for i, warehouse in enumerate(['仓库A(玉田)', '仓库B(辛集)', '仓库C(南宫)']):
    print(f'\n{warehouse}:')
    for j, city in enumerate(['石家庄', '唐山', '秦皇岛', '邯郸', '邢台', '保定', '张家口', '承德', '沧州', '廊坊', '衡水', '北京', '天津']):
        if optimal_solution[i, j] > 0:
            print(f'  到{city} ({optimal_solution[i, j]:.2f}万吨):')
            weight = optimal_solution[i, j]
            distance = distances[i][j]

            # 计算每种车型最多需要的数量
            max_vehicles = {}
            for vehicle in vehicle_types:
                max_vehicles[vehicle] = ceil(weight / vehicle_info[vehicle]['最大运量'])

            # 生成所有可能的车型组合
            possible_combinations = []
            for combo in product(
                range(max_vehicles[vehicle_types[0]] + 1),
                range(max_vehicles[vehicle_types[1]] + 1),
                range(max_vehicles[vehicle_types[2]] + 1)
            ):
                # 计算该组合的总运力
                total_capacity = (
                    combo[0] * vehicle_info[vehicle_types[0]]['最大运量'] +
                    combo[1] * vehicle_info[vehicle_types[1]]['最大运量'] +
                    combo[2] * vehicle_info[vehicle_types[2]]['最大运量']
                )

                # 如果总运力大于等于需要运输的重量，则该组合有效
                if total_capacity >= weight:
                    possible_combinations.append(combo)

            # 计算每种组合的总成本
            min_cost = float('inf')
            best_combo = None

            for combo in possible_combinations:
                remaining = weight
                cost = 0

                # 按车型容量从大到小分配
                vehicles_used = {
                    vehicle_types[0]: combo[0],
                    vehicle_types[1]: combo[1],
                    vehicle_types[2]: combo[2]
                }

                # 先使用重型车
                if vehicles_used['重型车'] > 0:
                    for _ in range(vehicles_used['重型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['重型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['重型车']['单位交通成本'] * distance * load +
                            vehicle_info['重型车']['单位油耗'] * distance
                        )
                        remaining -= load

                # 再使用中型车
                if vehicles_used['中型车'] > 0:
                    for _ in range(vehicles_used['中型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['中型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['中型车']['单位交通成本'] * distance * load +
                            vehicle_info['中型车']['单位油耗'] * distance
                        )
                        remaining -= load

                # 最后使用小型车
                if vehicles_used['小型车'] > 0:
                    for _ in range(vehicles_used['小型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['小型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['小型车']['单位交通成本'] * distance * load +
                            vehicle_info['小型车']['单位油耗'] * distance
                        )
                        remaining -= load

                # 更新最优方案
                if cost < min_cost:
                    min_cost = cost
                    best_combo = combo

            # 输出最优方案
            if best_combo:
                total_cost += min_cost
                total_trips += sum(best_combo)

                print('    最优方案:')
                for idx, vehicle in enumerate(vehicle_types):
                    if best_combo[idx] > 0:
                        print(f'    {vehicle}: {best_combo[idx]}车次')

                print(f'    成本: {min_cost:.2f}万元')

city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])
demand_satisfaction = np.sum(optimal_solution, axis=0) / city_demands

print(f'\n总运输车次: {total_trips}')
print(f'总运输成本: {total_cost:.2f}万元')
print('各地区需求满足率：', demand_satisfaction)
print('平均需求满足率：', np.mean(demand_satisfaction))

车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：

仓库A(玉田):
  到唐山 (30.00万吨):
    最优方案:
    重型车: 1车次
    成本: 519.15万元
  到秦皇岛 (9.07万吨):
    最优方案:
    小型车: 1车次
    成本: 482.10万元
  到承德 (14.96万吨):
    最优方案:
    小型车: 2车次
    成本: 805.00万元
  到廊坊 (14.63万吨):
    最优方案:
    小型车: 2车次
    成本: 679.24万元
  到北京 (10.00万吨):
    最优方案:
    小型车: 1车次
    成本: 387.31万元
  到天津 (6.34万吨):
    最优方案:
    小型车: 1车次
    成本: 285.74万元

仓库B(辛集):
  到石家庄 (30.00万吨):
    最优方案:
    重型车: 1车次
    成本: 658.52万元
  到保定 (40.00万吨):
    最优方案:
    重型车: 1车次
    成本: 1286.56万元
  到沧州 (39.91万吨):
    最优方案:
    重型车: 1车次
    成本: 1445.86万元
  到廊坊 (5.68万吨):
    最优方案:
    小型车: 1车次
    成本: 606.63万元

仓库C(南宫):
  到邯郸 (36.98万吨):
    最优方案:
    重型车: 1车次
    成本: 1272.14万元
  到邢台 (36.91万吨):
    最优方案:
    重型车: 1车次
    成本: 883.33万元
  到衡水 (30.00万吨):
    最优方案:
    重型车: 1车次
    成本: 448.25万元

总运输车次: 15
总运输成本: 9759.84万元
各地区需求满足率： [0.95122678 0.9941862  1.         1.         1.         0.9219993
 0.         1.         1.         1.         0.98732624 0.94966762
 1.        ]
平均需求满足率： 0.

In [ ]:
# 微调运输方案3
import numpy as np
from itertools import product
from math import ceil

# 初始方案的最优配送结果（万吨）
optimal_solution = np.array([
    [ 0.        , 30,  9.06583095394121,  0.        ,  0.        ,  0.        ,
      0.        , 14.9595917197154,  0.        , 14.6345773263434,  0.        , 10     ,
      6.34      ],
    [30,  0.        ,  0.        ,  0.        ,  0.        , 40,
      0.        ,  0.        , 39.9129516944393,  0,  0.        ,  0.        ,
      0.        ],
    [ 0.        ,  0.        ,  0.        , 36.9777190601011, 36.911715151657,  0.        ,
      0.        ,  0.        ,  0.        ,  0, 30,  0.        ,
      0.        ]
])

# 车型信息
vehicle_info = {
    '小型车': {'最大运量': 10, '单位交通成本': 0.145, '单位油耗': 1.4},
    '中型车': {'最大运量': 20, '单位交通成本': 0.0975, '单位油耗': 3.6},
    '重型车': {'最大运量': 40, '单位交通成本': 0.045, '单位油耗': 6.8}
}

# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])

vehicle_types = list(vehicle_info.keys())

print('车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：')
total_trips = 0
total_cost = 0

for i, warehouse in enumerate(['仓库A(玉田)', '仓库B(辛集)', '仓库C(南宫)']):
    print(f'\n{warehouse}:')
    for j, city in enumerate(['石家庄', '唐山', '秦皇岛', '邯郸', '邢台', '保定', '张家口', '承德', '沧州', '廊坊', '衡水', '北京', '天津']):
        if optimal_solution[i, j] > 0:
            print(f'  到{city} ({optimal_solution[i, j]:.2f}万吨):')
            weight = optimal_solution[i, j]
            distance = distances[i][j]

            # 计算每种车型最多需要的数量
            max_vehicles = {}
            for vehicle in vehicle_types:
                max_vehicles[vehicle] = ceil(weight / vehicle_info[vehicle]['最大运量'])

            # 生成所有可能的车型组合
            possible_combinations = []
            for combo in product(
                range(max_vehicles[vehicle_types[0]] + 1),
                range(max_vehicles[vehicle_types[1]] + 1),
                range(max_vehicles[vehicle_types[2]] + 1)
            ):
                # 计算该组合的总运力
                total_capacity = (
                    combo[0] * vehicle_info[vehicle_types[0]]['最大运量'] +
                    combo[1] * vehicle_info[vehicle_types[1]]['最大运量'] +
                    combo[2] * vehicle_info[vehicle_types[2]]['最大运量']
                )

                # 如果总运力大于等于需要运输的重量，则该组合有效
                if total_capacity >= weight:
                    possible_combinations.append(combo)

            # 计算每种组合的总成本
            min_cost = float('inf')
            best_combo = None

            for combo in possible_combinations:
                remaining = weight
                cost = 0

                # 按车型容量从大到小分配
                vehicles_used = {
                    vehicle_types[0]: combo[0],
                    vehicle_types[1]: combo[1],
                    vehicle_types[2]: combo[2]
                }

                # 先使用重型车
                if vehicles_used['重型车'] > 0:
                    for _ in range(vehicles_used['重型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['重型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['重型车']['单位交通成本'] * distance * load +
                            vehicle_info['重型车']['单位油耗'] * distance
                        )
                        remaining -= load

                # 再使用中型车
                if vehicles_used['中型车'] > 0:
                    for _ in range(vehicles_used['中型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['中型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['中型车']['单位交通成本'] * distance * load +
                            vehicle_info['中型车']['单位油耗'] * distance
                        )
                        remaining -= load

                # 最后使用小型车
                if vehicles_used['小型车'] > 0:
                    for _ in range(vehicles_used['小型车']):
                        if remaining <= 0:
                            break
                        load = min(vehicle_info['小型车']['最大运量'], remaining)
                        cost += (
                            vehicle_info['小型车']['单位交通成本'] * distance * load +
                            vehicle_info['小型车']['单位油耗'] * distance
                        )
                        remaining -= load

                # 更新最优方案
                if cost < min_cost:
                    min_cost = cost
                    best_combo = combo

            # 输出最优方案
            if best_combo:
                total_cost += min_cost
                total_trips += sum(best_combo)

                print('    最优方案:')
                for idx, vehicle in enumerate(vehicle_types):
                    if best_combo[idx] > 0:
                        print(f'    {vehicle}: {best_combo[idx]}车次')

                print(f'    成本: {min_cost:.2f}万元')

city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])
demand_satisfaction = np.sum(optimal_solution, axis=0) / city_demands

print(f'\n总运输车次: {total_trips}')
print(f'总运输成本: {total_cost:.2f}万元')
print('各地区需求满足率：', demand_satisfaction)
print('平均需求满足率：', np.mean(demand_satisfaction))

车型优化后的配送方案（考虑所有可能的车型组合，选择最低成本）：

仓库A(玉田):
  到唐山 (30.00万吨):
    最优方案:
    重型车: 1车次
    成本: 519.15万元
  到秦皇岛 (9.07万吨):
    最优方案:
    小型车: 1车次
    成本: 482.10万元
  到承德 (14.96万吨):
    最优方案:
    小型车: 2车次
    成本: 805.00万元
  到廊坊 (14.63万吨):
    最优方案:
    小型车: 2车次
    成本: 679.24万元
  到北京 (10.00万吨):
    最优方案:
    小型车: 1车次
    成本: 387.31万元
  到天津 (6.34万吨):
    最优方案:
    小型车: 1车次
    成本: 285.74万元

仓库B(辛集):
  到石家庄 (30.00万吨):
    最优方案:
    重型车: 1车次
    成本: 658.52万元
  到保定 (40.00万吨):
    最优方案:
    重型车: 1车次
    成本: 1286.56万元
  到沧州 (39.91万吨):
    最优方案:
    重型车: 1车次
    成本: 1445.86万元

仓库C(南宫):
  到邯郸 (36.98万吨):
    最优方案:
    重型车: 1车次
    成本: 1272.14万元
  到邢台 (36.91万吨):
    最优方案:
    重型车: 1车次
    成本: 883.33万元
  到衡水 (30.00万吨):
    最优方案:
    重型车: 1车次
    成本: 448.25万元

总运输车次: 14
总运输成本: 9153.21万元
各地区需求满足率： [0.95122678 0.9941862  1.         1.         1.         0.9219993
 0.         1.         1.         0.7205676  0.98732624 0.94966762
 1.        ]
平均需求满足率： 0.8865364419045173
